In [ ]:
!pip install --upgrade transformers --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 48.0 MB/s eta 0:00:00


In [ ]:
import transformers
print(transformers.__version__)


4.53.1


In [ ]:

# -------------------- 📚 Load Dataset Correctly --------------------
from datasets import Dataset
import json

file_path = "/content/cricket_qa_dataset_real.jsonl"

with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f.readlines()]

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.1)


# -------------------- 🔧 Load T5 Model & Tokenizer --------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# -------------------- 🛠️ Preprocess Dataset --------------------
def preprocess_function(examples):
    inputs = examples['input']
    targets = examples['output']
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)


# -------------------- ⚙️ Training Configuration --------------------
training_args = TrainingArguments(
    output_dir="./t5_cricket_finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_dir="./logs",
    report_to="none",  # 👈 This disables wandb properly
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)


# -------------------- 🚀 Fine-Tune the Model --------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)


trainer.train()


# -------------------- 💾 Save Fine-Tuned Model --------------------
model.save_pretrained("./t5_cricket_chatbot")
tokenizer.save_pretrained("./t5_cricket_chatbot")


# -------------------- 🗣️ Inference Chatbot --------------------
from transformers import pipeline

pipe = pipeline("text2text-generation", model="./t5_cricket_chatbot", tokenizer="./t5_cricket_chatbot")

print("Chatbot is ready! Type 'exit' to stop.")
while True:
    question = input("Ask me about cricket: ")
    if question.lower() in ["exit", "quit"]:
        break
    output = pipe(question, max_length=100)
    print("Answer:", output[0]['generated_text'])


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss


Device set to use cpu


...

Answer: Sachin Tendulkar
Ask me about cricket: what is ICC stands for


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: International Cricket Council
Ask me about cricket: who was captain of india in 2011 odi world cup


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Sachin Tendulkar
Ask me about cricket: exit


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained("/content/drive/MyDrive/t5_cricket_chatbot")
tokenizer.save_pretrained("/content/drive/MyDrive/t5_cricket_chatbot")


Mounted at /content/drive


('/content/drive/MyDrive/t5_cricket_chatbot/tokenizer_config.json',
 '/content/drive/MyDrive/t5_cricket_chatbot/special_tokens_map.json',
 '/content/drive/MyDrive/t5_cricket_chatbot/spiece.model',
 '/content/drive/MyDrive/t5_cricket_chatbot/added_tokens.json',
 '/content/drive/MyDrive/t5_cricket_chatbot/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------- 📥 Load Fine-Tuned Model --------------------
model = AutoModelForSeq2SeqLM.from_pretrained("./t5_cricket_chatbot")
tokenizer = AutoTokenizer.from_pretrained("./t5_cricket_chatbot")


# -------------------- 💬 Chat Function --------------------
def ask_cricket_bot(question):
    input_text = f"Question: {question} Answer:"
    inputs = tokenizer(input_text, return_tensors="pt")

    outputs = model.generate(**inputs, max_length=100)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


# -------------------- 🤖 Chat Example --------------------
while True:
    question = input("\nAsk Cricket Bot: ")
    if question.lower() in ["exit", "quit"]:
        break
    answer = ask_cricket_bot(question)
    print(f"🏏 Cricket Bot: {answer}")



Ask Cricket Bot:  which country won 2007 t20 world cup
🏏 Cricket Bot: India

Ask Cricket Bot: exit
